In [ ]:
import numpy as np
import time as time
import os
import copy

import lyafxcorr_kg as xcorr
import constants
constants.DATA_VERSION = 'v0'

import pandas as pd
# Set up matplotlib and use a nicer set of plot parameters
%config InlineBackend.rc = {}
import matplotlib as mpl
mpl.rc('mathtext',fontset='stixsans')
mpl.rc('figure', facecolor="white")
#matplotlib.rc_file("../../templates/matplotlibrc")
import matplotlib.pyplot as plt
#import matplotlib.colors as colors
%matplotlib inline

import astropy.table
from astropy.cosmology import FlatLambdaCDM
from astropy.io import fits
from astropy.io import ascii
from astropy.table import Table
from astropy import units as u
from astropy.coordinates import SkyCoord

import plotting
plotting.plot_preamble()

def taueff_evo(z):
    return 0.001845 * (1.+z)**3.924

# Define cosmology
cosmo = constants.COSMOLOGY
zmin = 2.0
zmid = 2.3
comdist_mean = cosmo.comoving_distance(zmid)
comdist_zmin = cosmo.comoving_distance(zmin)
dcomdist_dz = cosmo.inv_efunc(zmid) *2998. # in Mpc/h
sim_boxlen_mpch = 250

dz_to_dmpch = lambda z: z * dcomdist_dz
dkms_to_dmpch = lambda v: ((v * u.km / u.s) / cosmo.H(zmid)).value * cosmo.h * (1 + zmid)

In [ ]:
specz_cat = Table.read(os.path.join(constants.GAL_DIR_BASE, 'mosdef_zcat.final.fits'))

In [ ]:
np.unique(specz_cat['Z_MOSFIRE_ZQUAL'], return_counts=True)

In [ ]:
specz_cat = ascii.read(os.path.join(constants.GAL_DIR_BASE, 'all_specz_v3_comb_COSMOS2020_v3.dat'))

In [ ]:
specz_cat

In [ ]:
cosmos_compilation_smass = Table.read(os.path.join(constants.GAL_DIR_BASE, '..', 'cosmos-speczcompilation', 'lephare_results_specz_compilation_DR1.1.fits'))

In [ ]:
cosmos_compilation_smass = cosmos_compilation_smass[['Id', 'mass_best', 'mass_med']]
cosmos_compilation_smass.rename_column('Id', 'Id_COS20_Classic')
cosmos_compilation_smass.rename_column('mass_best', 'Ms_best')
cosmos_compilation_smass.rename_column('mass_med', 'Ms_med')

cosmos_compilation_smass['Id_COS20_Classic'] = cosmos_compilation_smass['Id_COS20_Classic'].astype(np.int64)

# Filter out entries with missing IDs (-999).
cosmos_compilation_smass = cosmos_compilation_smass[cosmos_compilation_smass['Id_COS20_Classic'] != -999]

In [ ]:
cosmos_compilation = Table.read(os.path.join(constants.GAL_DIR_BASE, '..', 'cosmos-speczcompilation', 'specz_compilation_COSMOS_DR1.1_unique.fits'))
print(cosmos_compilation.columns)
cosmos_compilation = cosmos_compilation[['Id_COS20_Classic', 'Id_COS20_Farmer', 'Confidence_level', 'flag']]

# Filter out entries with missing IDs (-999).
cosmos_compilation = cosmos_compilation[cosmos_compilation['Id_COS20_Classic'] != -999]
cosmos_compilation = cosmos_compilation[cosmos_compilation['Id_COS20_Farmer'] != -999]

In [ ]:
cosmos_compilation = astropy.table.join(cosmos_compilation, cosmos_compilation_smass, keys='Id_COS20_Classic', join_type='left')
cosmos_compilation.rename_column('Id_COS20_Farmer', 'id')
del cosmos_compilation['Id_COS20_Classic']

In [ ]:
new_specz_cat = specz_cat.copy()
del new_specz_cat['Ms_best']
del new_specz_cat['Ms_med']

new_specz_cat = astropy.table.join(new_specz_cat, cosmos_compilation, keys='id', join_type='left')

included_surveys = ['CLAMATO', 'MOSDEF', 'VUDS', 'zDeep']

new_specz_cat = new_specz_cat[np.isin(new_specz_cat['source'], included_surveys)]

new_specz_cat['ID_specz'] = new_specz_cat['ID_specz'].astype(np.int64)

# Match VUDS DR2

On 3/2/26, KG sent me this catalog; might have quality flags for VUDS DR2 in it?

In [ ]:
# specz_cat = ascii.read(os.path.join(constants.GAL_DIR_BASE, 'all_specz_v3_comb_COSMOS2020_v3.dat'))

# vuds_cat = specz_cat[specz_cat['source'] == 'VUDS']
# print(len(vuds_cat))
# vuds_cat['ID_specz'] = vuds_cat['ID_specz'].astype(np.int64)

cesam_vuds_cat = ascii.read(os.path.join(constants.GAL_DIR_BASE, 'cesam_vuds_spectra_cosmos_catalog_1489008861.txt'))
cesam_vuds_cat.rename_column('ident', 'ID_specz')
cesam_vuds_cat.rename_column('zflags', 'vuds_zflags')
cesam_vuds_cat['ID_specz'] = cesam_vuds_cat['ID_specz'].astype(np.int64)
cesam_vuds_cat = cesam_vuds_cat[['ID_specz', 'vuds_zflags']]

In [ ]:
new_specz_cat = astropy.table.join(new_specz_cat, cesam_vuds_cat, keys='ID_specz', join_type='left')

In [ ]:
new_specz_cat

In [ ]:
# If confidence level is missing and vuds_zflags are available, write new confidence level into catalog using midpoint of range.
new_conf_levels = 0
for i, (row, cf_mask, zf_mask) in enumerate(zip(new_specz_cat, new_specz_cat.mask['Confidence_level'], new_specz_cat.mask['vuds_zflags'])):
    if cf_mask and not zf_mask:
        # https://doi.org/10.1051/0004-6361/201527963
        # if not row['flag']:
        row['flag'] = row['vuds_zflags']
        if row['vuds_zflags'] == 4:
            row['Confidence_level'] = 100
        elif row['vuds_zflags'] == 3:
            row['Confidence_level'] = 95
        elif row['vuds_zflags'] == 2:
            row['Confidence_level'] = 80
        elif row['vuds_zflags'] == 1:
            row['Confidence_level'] = 50
        elif row['vuds_zflags'] == 9:
            row['Confidence_level'] = 85
        else:
            new_conf_levels -= 1
        new_conf_levels += 1

print(new_conf_levels)

In [ ]:
new_specz_cat

In [ ]:
# ascii.write(new_specz_cat, os.path.join(constants.GAL_DIR_BASE, 'all_specz_v3_comb_COSMOS_compilation_v0.dat'))

In [ ]:
new_specz_cat['Ms_best'].info

In [ ]:

for s in included_surveys:
    print(s, len(specz_cat[specz_cat['source'] == s]))

print()

new_specz_cat_no3dhst = new_specz_cat[np.isin(new_specz_cat['source'], included_surveys)]
print(len(new_specz_cat_no3dhst))
survey_names = sorted(list(set(new_specz_cat_no3dhst['source'])))
source_sep = [new_specz_cat_no3dhst[new_specz_cat_no3dhst['source'] == s]['Confidence_level'] for s in survey_names]

for s, c in zip(included_surveys, source_sep):
    print(s, len(c))
    if s == 'VUDS':
        vuds_ids_in_cosmos_comp = new_specz_cat_no3dhst[new_specz_cat_no3dhst["source"] == s]["ID_specz"]
        print(f'VUDS IDs are {vuds_ids_in_cosmos_comp}')
        print(c)
plt.hist(source_sep, stacked=True, label=survey_names, bins=50)
plt.legend()
plt.xlabel('Confidence level')
plt.ylabel('# galaxies')

In [ ]:
from collections import defaultdict

smass_bin_boundaries = [-np.inf, 9.53845, 9.94789, np.inf]

vuds_subcat = new_specz_cat_no3dhst[new_specz_cat_no3dhst['source'] == 'VUDS']

split_vuds = []
for i in range(3):
    sc = new_specz_cat_no3dhst[np.logical_and(new_specz_cat_no3dhst['Ms_best'] > smass_bin_boundaries[i], new_specz_cat_no3dhst['Ms_best'] <= smass_bin_boundaries[i + 1])]['Confidence_level']
    split_vuds.append(sc)

bottom = defaultdict(int)
for cl_data, label, color in zip(split_vuds, ['VUDS LM', 'VUDS MM', 'VUDS HM'], ['cornflowerblue', 'orange', 'red']):
    uniq_cl, cl_counts = np.unique(cl_data, return_counts=True)
    uniq_cl = np.array(uniq_cl)
    print(uniq_cl, cl_counts)
    unpacked_bottom = [bottom[ucl] for ucl in uniq_cl]
    plt.gca().bar(uniq_cl, cl_counts, width=1, label=label, color=color, bottom=unpacked_bottom)
    for ucl, c in zip(uniq_cl, cl_counts):
        bottom[ucl] += c

#plt.hist(split_vuds, stacked=True, label=, , bins=20, range=(70, 100));
plt.xlabel('Spec-z confidence level')
plt.ylabel('# galaxies')
plt.xlim(68, 102)
plt.legend()
plt.savefig('vuds-redshift-confidence.png')

vuds_lm = split_vuds[0]
vuds_mm = split_vuds[1]
vuds_hm = split_vuds[2]

print(f'For VUDS low mass, {np.sum(vuds_lm < 90)/len(vuds_lm):.3f} proportion with <90% CL and {np.sum(vuds_lm >= 90)/len(vuds_lm):.3f} proportion with >=90%. {np.sum(vuds_lm.mask)} missing confidence levels')
print(f'For VUDS medium mass, {np.sum(vuds_mm < 90)/len(vuds_mm):.3f} proportion with <90% CL and {np.sum(vuds_mm >= 90)/len(vuds_mm):.3f} proportion with >=90%. {np.sum(vuds_mm.mask)} missing confidence levels')
print(f'For VUDS high mass, {np.sum(vuds_hm < 90)/len(vuds_hm):.3f} proportion with <90% CL and {np.sum(vuds_hm >= 90)/len(vuds_hm):.3f} proportion with >=90%. {np.sum(vuds_hm.mask)} missing confidence levels')

In [ ]:
from collections import defaultdict

smass_bin_boundaries = [-np.inf, 9.53845, 9.94789, np.inf]

vuds_subcat = new_specz_cat_no3dhst[new_specz_cat_no3dhst['source'] == 'VUDS']

split_vuds = []
for i in range(3):
    sc = new_specz_cat_no3dhst[np.logical_and(new_specz_cat_no3dhst['Ms_best'] > smass_bin_boundaries[i], new_specz_cat_no3dhst['Ms_best'] <= smass_bin_boundaries[i + 1])]['flag']
    split_vuds.append(sc)

bottom = defaultdict(int)
for cl_data, label, color in zip(split_vuds, ['VUDS LM', 'VUDS MM', 'VUDS HM'], ['cornflowerblue', 'orange', 'red']):
    uniq_cl, cl_counts = np.unique(cl_data, return_counts=True)
    uniq_cl = np.array(uniq_cl)
    print(uniq_cl, cl_counts)
    unpacked_bottom = [bottom[ucl] for ucl in uniq_cl]
    plt.gca().bar(uniq_cl, cl_counts, width=0.2, label=label, color=color, bottom=unpacked_bottom)
    for ucl, c in zip(uniq_cl, cl_counts):
        bottom[ucl] += c

#plt.hist(split_vuds, stacked=True, label=, , bins=20, range=(70, 100));
plt.xlabel('Spec-z quality flag')
plt.ylabel('# galaxies')
plt.xlim(1.5, 10)
plt.legend()
plt.savefig('vuds-redshift-flags.png')

# Compare spec-z diffs

In [ ]:
cosmos_compilation_nonuniq = Table.read(os.path.join(constants.GAL_DIR_BASE, '..', 'cosmos-speczcompilation', 'specz_compilation_COSMOS_DR1.1_all.fits'))

In [ ]:
cosmos_compilation_uniq = Table.read(os.path.join(constants.GAL_DIR_BASE, '..', 'cosmos-speczcompilation', 'specz_compilation_COSMOS_DR1.1_unique.fits'))

In [ ]:
print(cosmos_compilation_nonuniq.columns)

In [ ]:
cosmos_compilation_uniq = cosmos_compilation_uniq[['Id_specz', 'specz', 'Confidence_level']]
cosmos_compilation_uniq.rename_column('specz', 'specz_best')
cosmos_compilation_uniq.rename_column('Confidence_level', 'Confidence_level_best')

cosmos_compilation_nonuniq = astropy.table.join(cosmos_compilation_nonuniq, cosmos_compilation_uniq, keys='Id_specz', join_type='left')

In [ ]:
cosmos_compilation_only_dupes = cosmos_compilation_nonuniq[cosmos_compilation_nonuniq['GroupID'] != -1]
print(len(np.unique(cosmos_compilation_only_dupes['GroupID'])))
print(len(cosmos_compilation_nonuniq), len(cosmos_compilation_only_dupes))

In [ ]:
# From README at https://github.com/cosmosastro/speczcompilation/blob/main/README.md,
# we're interested in redshifts with confidence >= 75%, which corresponds to 2-4 & 9 in the unified quality flag system
#allowed_quality = (2, 3, 4, 9)
CONF_THRES = 75

cosmos_compilation_only_dupes = cosmos_compilation_only_dupes[cosmos_compilation_only_dupes['Confidence_level'] > CONF_THRES]
print(len(cosmos_compilation_only_dupes))

In [ ]:
specz_diff_from_best = []

for grp in np.unique(cosmos_compilation_only_dupes['GroupID']):
    subset = cosmos_compilation_only_dupes[cosmos_compilation_only_dupes['GroupID'] == grp]
    if len(subset) == 1:
        continue
    best_entry = subset[~subset['specz_best'].mask]
    if len(best_entry) == 0:
        continue
    best_entry = best_entry[0]
    # if np.abs(np.max(subset['specz'] - best_entry['specz_best'])) > 1:
    #     print(subset[['specz', 'Confidence_level', 'specz_best', 'Confidence_level_best']])
    #     print()
    for row in subset[subset['specz_best'].mask]:
        specz_diff_from_best.append((best_entry['specz_best'], best_entry['Confidence_level_best'],
            row['Confidence_level'] - best_entry['Confidence_level_best'],
            row['specz'] - best_entry['specz_best']))

specz_diff_from_best = np.array(specz_diff_from_best)

In [ ]:
np.unique(specz_diff_from_best[:, 1])

In [ ]:
subset_mask = np.logical_and.reduce([specz_diff_from_best[:, 0] > 2,
                                    specz_diff_from_best[:, 0] < 3,
                                    specz_diff_from_best[:, 1] >= 95,
                                    specz_diff_from_best[:, 1] <= 97,
                                    np.abs(specz_diff_from_best[:, 3]) < 10])

subset = specz_diff_from_best[subset_mask]

In [ ]:
plt.hist(np.abs(subset[:, 3]))

In [ ]:
plt.hexbin(subset[:, 2], np.log10(np.abs(subset[:, 3])), gridsize=30, vmax=50)
plt.xlabel('Delta confidence level from highest-confidence measurement')
plt.ylabel('log10(|spec-z - best spec-z|)')
plt.title('2 <= Best spec-z <= 3, best spec-z confidence >= 95%')
plt.colorbar()

In [ ]:
subset

In [ ]:
z_conf_stddev = []

for grp in np.unique(cosmos_compilation_only_dupes['GroupID']):
    subset = cosmos_compilation_only_dupes[cosmos_compilation_only_dupes['GroupID'] == grp]
    if len(subset) == 1:
        continue
    z_conf_stddev.append((np.mean(subset['specz']), np.mean(subset['Confidence_level']), np.std(subset['specz'])))

In [ ]:
z_conf_stddev = np.array(z_conf_stddev)
z_conf_stddev = z_conf_stddev[z_conf_stddev[:, 2] != 0]

In [ ]:
z_conf_stddev[:, 2].min()

In [ ]:
plt.hist(np.log10(z_conf_stddev[:, 2]))

In [ ]:
import matplotlib.colors as colors
plt.hexbin(z_conf_stddev[:, 1], z_conf_stddev[:, 0], C=z_conf_stddev[:, 2], gridsize=50,
           norm=colors.LogNorm(vmin=1e-5, vmax=z_conf_stddev[:, 2].max()),
           reduce_C_function=np.median)
plt.xlabel('Mean confidence level')
plt.ylabel('Mean spec-z')
plt.colorbar(label='Median of spec-z std. dev')

In [ ]:
cosmos_compilation_surveys = Table.read(os.path.join(constants.GAL_DIR_BASE, '..', 'cosmos-speczcompilation', 'specz_compilation_COSMOS_DR1.1_surveys.list'), format='ascii')

survey_id_mapping = {n: int(i) for i, n in zip(cosmos_compilation_surveys['ID'], cosmos_compilation_surveys['Survey'])}

In [ ]:
survey_id_mapping

In [ ]:
SURVEY_SUBSET = ('CLAMATODR2', 'MOSDEF', 'VUDSDR1', 'VUDSDR2', 'zCOSMOSD')